Name: Muhammad Raziq Bin Sufian

Id: 24006626

## 1) Storage Method: InfluxDB (Time-Series Database)

**Justification:** Every digital twin update such as sensor reading  is written as its own InfluxDB point, using the variable's name as the measurement for example motor_temperature and tags "twin_id" and data_type. Numeric values go in field `value_num`, text values in field `value_str`, since InfluxDB fields must be a single type.

This keeps the schema generic and future-proof: a new variable can be streamed in at any time with no structural change, unlike a relational table which needs a new column per variable or a flat file which have no querying by time or by twin. InfluxDB also gives both things a digital twin needs from one store: the latest value of a variable and its full history over time (a time-range query).

## 2) State Variables & Sensor Streams (twin: robot_001)

**State variables** (`data_type = "state"`) — describe current condition, change in discrete steps, only the latest value usually matters:
- `operational_status` (idle / moving / charging) — what the robot is doing right now.
- `current_zone` (Zone_A / Zone_B) — where the robot currently is, used for zone-based rules.

**Sensor streams** (`data_type = "sensor"`) — continuous numeric readings, meaningful as a history over time:
- `motor_temperature` — tracked over time to catch overheating trends.
- `battery_voltage` — tracked over time to monitor power/charge degradation.

Both are stored the same way, but queried differently: state is read with `last()` which only the current value matters, sensors are read over a time range which the trend matters.

## 3) Kafka as Stream Manager

**Justification:** Kafka sits between the digital twin (producer) and InfluxDB (consumer). The producer only needs to know the topic name, not the database schema or connection — the consumer can be restarted or replaced independently, and other consumers such as an alerting service could subscribe to the same topic later with no change to the producer.

**Route:** Local broker at `127.0.0.1:9092`, topic `digitaltwin.telemetry`. Each message is a generic JSON envelope `{twin_id, type, measurement_name, value}`, published by the producer and picked up by a background consumer thread, which writes it into InfluxDB as the measurement named `measurement_name`, tagged by `twin_id` and `type`.

## 4) Install dependencies

In [1]:
%pip install influxdb-client kafka-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 5) Connect to InfluxDB

In [2]:
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS
import json, time, threading
from kafka import KafkaProducer, KafkaConsumer

INFLUX_URL = "http://localhost:8086"
INFLUX_TOKEN = "fJkA8iV64KKnTXkuuzLGHLpVBIDXBAWGlm3eyZFteWjECM1Bc89nx_5ikhNerb5fg_hjAyZ6CuVo4OMAzSAJTw=="
INFLUX_ORG = "raziq"
INFLUX_BUCKET = "digitaltwin"

client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
write_api = client.write_api(write_options=SYNCHRONOUS)
query_api = client.query_api()

buckets_api = client.buckets_api()
if not buckets_api.find_bucket_by_name(INFLUX_BUCKET):
    buckets_api.create_bucket(bucket_name=INFLUX_BUCKET, org=INFLUX_ORG)
    print(f"Created bucket '{INFLUX_BUCKET}'")
else:
    print(f"Bucket '{INFLUX_BUCKET}' already exists")

print("InfluxDB reachable:", client.ping())

Bucket 'digitaltwin' already exists
InfluxDB reachable: True


## 6) Kafka consumer: write incoming messages into InfluxDB

In [3]:
KAFKA_BROKER = "127.0.0.1:9092"
KAFKA_TOPIC = "digitaltwin.telemetry"
consumer_ready = threading.Event()

def write_message_to_influx(msg):
    point = Point(msg["measurement_name"]) \
        .tag("twin_id", msg["twin_id"]) \
        .tag("data_type", msg["type"])

    if isinstance(msg["value"], (int, float)):
        point.field("value_num", float(msg["value"]))
    else:
        point.field("value_str", str(msg["value"]))

    write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG, record=point)

def consume_loop():
    consumer = KafkaConsumer(
        KAFKA_TOPIC,
        bootstrap_servers=KAFKA_BROKER,
        auto_offset_reset="latest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        group_id=f"digitaltwin-storage-writer-{int(time.time())}"
    )
    consumer.poll(timeout_ms=1000)
    consumer_ready.set()

    while True:
        try:
            for message in consumer:
                msg = message.value
                try:
                    write_message_to_influx(msg)
                    print(f"[KAFKA CONSUMER] Stored {msg['type']} '{msg['measurement_name']}' = {msg['value']} for {msg['twin_id']}")
                except Exception as e:
                    print(f"[KAFKA CONSUMER ERROR] Failed to write {msg}: {e}")
        except Exception as e:
            print(f"[KAFKA CONSUMER FATAL] Consumer loop crashed, restarting: {e}")
            time.sleep(1)

consumer_thread = threading.Thread(target=consume_loop, daemon=True)
consumer_thread.start()

if consumer_ready.wait(timeout=10):
    print("Kafka consumer subscribed and ready.")
else:
    print("Warning: consumer did not confirm readiness in time.")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25584\928384239.py:18: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Kafka consumer subscribed and ready.


## 7) Publish initial digital twin data



In [4]:
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

initial_updates = [
    {"twin_id": "robot_001", "type": "sensor", "measurement_name": "motor_temperature", "value": 68.5},
    {"twin_id": "robot_001", "type": "sensor", "measurement_name": "battery_voltage", "value": 24.1},
    {"twin_id": "robot_001", "type": "state", "measurement_name": "operational_status", "value": "idle"},
    {"twin_id": "robot_001", "type": "state", "measurement_name": "current_zone", "value": "Zone_A"},
]

print("Publishing initial data...")
for update in initial_updates:
    producer.send(KAFKA_TOPIC, update)
producer.flush()
time.sleep(3)
print("Done.")

Publishing initial data...
[KAFKA CONSUMER] Stored sensor 'motor_temperature' = 68.5 for robot_001
[KAFKA CONSUMER] Stored sensor 'battery_voltage' = 24.1 for robot_001
[KAFKA CONSUMER] Stored state 'operational_status' = idle for robot_001
[KAFKA CONSUMER] Stored state 'current_zone' = Zone_A for robot_001
Done.


## 8) Update the digital twin

This demonstrates that a state variable can change and a sensor stream can grow: `operational_status` changes from `idle` to `moving`, `current_zone` changes from `Zone_A` to `Zone_B`, and two more `motor_temperature` / `battery_voltage` readings are added a couple of seconds apart so the sensor history has more than one point.

In [5]:
update_1 = [
    {"twin_id": "robot_001", "type": "state", "measurement_name": "operational_status", "value": "moving"},
    {"twin_id": "robot_001", "type": "state", "measurement_name": "current_zone", "value": "Zone_B"},
    {"twin_id": "robot_001", "type": "sensor", "measurement_name": "motor_temperature", "value": 71.2},
    {"twin_id": "robot_001", "type": "sensor", "measurement_name": "battery_voltage", "value": 23.8},
]
print("Publishing update 1...")
for update in update_1:
    producer.send(KAFKA_TOPIC, update)
producer.flush()
time.sleep(3)

update_2 = [
    {"twin_id": "robot_001", "type": "sensor", "measurement_name": "motor_temperature", "value": 73.9},
    {"twin_id": "robot_001", "type": "sensor", "measurement_name": "battery_voltage", "value": 23.5},
]
print("Publishing update 2...")
for update in update_2:
    producer.send(KAFKA_TOPIC, update)
producer.flush()
time.sleep(3)
print("Done. operational_status/current_zone updated once, sensors now have 3 points each.")

Publishing update 1...
[KAFKA CONSUMER] Stored state 'operational_status' = moving for robot_001
[KAFKA CONSUMER] Stored state 'current_zone' = Zone_B for robot_001
[KAFKA CONSUMER] Stored sensor 'motor_temperature' = 71.2 for robot_001
[KAFKA CONSUMER] Stored sensor 'battery_voltage' = 23.8 for robot_001
Publishing update 2...
[KAFKA CONSUMER] Stored sensor 'motor_temperature' = 73.9 for robot_001
[KAFKA CONSUMER] Stored sensor 'battery_voltage' = 23.5 for robot_001
Done. operational_status/current_zone updated once, sensors now have 3 points each.


## 9) Retrieve and validate: latest state vs. sensor history


In [6]:
print("--- Latest STATE values ---")
state_query = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -10m)
  |> filter(fn: (r) => r["twin_id"] == "robot_001" and r["data_type"] == "state")
  |> last()
'''
for table in query_api.query(org=INFLUX_ORG, query=state_query):
    for record in table.records:
        ts = record.get_time().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{ts}] {record.values.get('_measurement'):<20} = {record.get_value()}")

print()
print("--- Full SENSOR history ---")
sensor_query = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -10m)
  |> filter(fn: (r) => r["twin_id"] == "robot_001" and r["data_type"] == "sensor")
  |> sort(columns: ["_time"])
'''
for table in query_api.query(org=INFLUX_ORG, query=sensor_query):
    for record in table.records:
        ts = record.get_time().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{ts}] {record.values.get('_measurement'):<20} = {record.get_value()}")

--- Latest STATE values ---
[2026-07-04 15:32:28] current_zone         = Zone_B
[2026-07-04 15:32:28] operational_status   = moving

--- Full SENSOR history ---
[2026-07-04 15:32:08] battery_voltage      = 24.1
[2026-07-04 15:32:28] battery_voltage      = 23.8
[2026-07-04 15:32:31] battery_voltage      = 23.5
[2026-07-04 15:32:08] motor_temperature    = 68.5
[2026-07-04 15:32:28] motor_temperature    = 71.2
[2026-07-04 15:32:31] motor_temperature    = 73.9
